In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
from glob import glob
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
import warnings
import os
import openslide

warnings.filterwarnings('ignore')

# Device configuration
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def create_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

params = {
    'input_size': 512,
    'model_epoch': 99,
    'img_form': 'ndpi',
    'target_mpp': 2.0  # Target MPP for the model (trained at this resolution)
}
model_dir = '../../model/HE_IHC_translation/internal_ss/PD-L1'
result_dir = '../../results/HE_IHC_translation/internal_ss/PD-L1/wsi_results'
create_dir(result_dir)

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, features):
        super(ResidualBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Conv2d(features, features, kernel_size=3, padding=1),
            nn.InstanceNorm2d(features),
            nn.ReLU(inplace=True),
            nn.Conv2d(features, features, kernel_size=3, padding=1),
            nn.InstanceNorm2d(features),
            nn.Dropout(0.5)
        )

    def forward(self, x):
        return x + self.block(x)

class Generator(nn.Module):
    def __init__(self, input_channels, output_channels, n_residual_blocks=9):
        super(Generator, self).__init__()
        model = [
            nn.Conv2d(input_channels, 64, kernel_size=7, padding=3),
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace=True)
        ]

        # Downsampling
        in_features = 64
        out_features = in_features * 2
        for _ in range(4):
            model += [
                nn.Conv2d(in_features, out_features, kernel_size=3, stride=2, padding=1),
                nn.InstanceNorm2d(out_features),
                nn.ReLU(inplace=True)
            ]
            in_features = out_features
            out_features = in_features * 2

        # Residual blocks
        for _ in range(n_residual_blocks):
            model += [ResidualBlock(in_features)]

        # Upsampling
        out_features = in_features // 2
        for _ in range(4):
            model += [
                nn.ConvTranspose2d(in_features, out_features, kernel_size=3, stride=2, padding=1, output_padding=1),
                nn.InstanceNorm2d(out_features),
                nn.ReLU(inplace=True)
            ]
            in_features = out_features
            out_features = in_features // 2

        # Output layer
        model += [nn.Conv2d(64, output_channels, kernel_size=7, padding=3), nn.Tanh()]
        self.model = nn.Sequential(*model)

    def forward(self, x):
        return self.model(x)

print("✓ Generator architecture defined")

# Initialize and load model
F = Generator(3, 3).to(device)
model_path = f'{model_dir}/F_{params["model_epoch"]}.pth'
F.load_state_dict(torch.load(model_path, map_location=device))
F.eval()

print(f"✓ Loaded model: {model_path}")
print(f"✓ Generator F (IHC → HE) ready for testing")

In [ ]:
import re

data_dir_ihc = '../../data/IHC_HE_Pair_Data_GA_SS/PD-L1(22C3)/'
data_dir_hne = '../../data/IHC_HE_Pair_Data_GA_SS/PD-L1(HnE)/'
data_list = glob(f'{data_dir_ihc}*.{params["img_form"]}')
print(f"Found {len(data_list)} IHC WSI files")

# Build HnE lookup by sample ID (SS-XXXXX)
hne_files = glob(f'{data_dir_hne}*.{params["img_form"]}')
hne_dict = {}
for f in hne_files:
    m = re.search(r'SS-(\d+)', os.path.basename(f))
    if m:
        hne_dict[m.group(1)] = f

# Select one WSI pair
wsi_path = data_list[0]
sample_id = re.search(r'SS-(\d+)', os.path.basename(wsi_path)).group(1)
hne_path = hne_dict[sample_id]
print(f"IHC: {os.path.basename(wsi_path)}")
print(f"HnE: {os.path.basename(hne_path)}")

# Open both WSIs
slide = openslide.OpenSlide(wsi_path)
slide_hne = openslide.OpenSlide(hne_path)

native_mpp = float(slide.properties.get('openslide.mpp-x', 0.25))
print(f"\nIHC - Dimensions: {slide.dimensions}, MPP: {native_mpp:.4f}")
print(f"HnE - Dimensions: {slide_hne.dimensions}")

## 3. Tissue Mask from Thumbnail (HSI-based)

Convert the WSI thumbnail to HSV color space and use the saturation channel to create a tissue mask. Tissue regions have higher saturation than white background.

In [ ]:
from skimage.color import rgb2hsv
from skimage.morphology import binary_closing, binary_opening, disk, remove_small_objects
from skimage.filters import threshold_otsu
from scipy.ndimage import binary_fill_holes

patch_size = params['input_size']  # 512
downsample_factor = params['target_mpp'] / native_mpp

# Size to read at level 0 that corresponds to one patch at target MPP
read_size = int(patch_size * downsample_factor)

# Find the best level to read from
best_level = slide.get_best_level_for_downsample(downsample_factor)
level_downsample = slide.level_downsamples[best_level]
level_read_size = int(read_size / level_downsample)

W, H = slide.dimensions

# --- Overlap grid: 모든 좌표를 인덱스 기반으로 통일 ---
overlap = patch_size // 4  # 128px
stride = patch_size - overlap  # 384px (canvas space)
read_stride = int(stride * downsample_factor)  # level-0 space stride

# Level-0 positions
positions_x = list(range(0, W - read_size + 1, read_stride))
positions_y = list(range(0, H - read_size + 1, read_stride))
n_patches_x = len(positions_x)
n_patches_y = len(positions_y)

# Canvas size: 인덱스 기반으로 정확히 계산
out_w = (n_patches_x - 1) * stride + patch_size
out_h = (n_patches_y - 1) * stride + patch_size

# Canvas가 커버하는 level-0 영역
canvas_l0_w = positions_x[-1] + read_size
canvas_l0_h = positions_y[-1] + read_size

print(f"Downsample factor: {downsample_factor:.4f}")
print(f"Overlap: {overlap}px, Stride: {stride}px")
print(f"Grid: {n_patches_x} x {n_patches_y} = {n_patches_x * n_patches_y} patches")
print(f"Canvas: {out_w} x {out_h} px")
print(f"Level-0 coverage: {canvas_l0_w} x {canvas_l0_h}")

# --- Tissue mask: canvas 영역 기준으로 slide에서 직접 생성 ---
mask_level = slide.get_best_level_for_downsample(canvas_l0_w / (out_w // 4))
mask_level_ds = slide.level_downsamples[mask_level]
mask_read_w = int(canvas_l0_w / mask_level_ds)
mask_read_h = int(canvas_l0_h / mask_level_ds)
mask_region = slide.read_region((0, 0), mask_level, (mask_read_w, mask_read_h))
mask_region_np = np.array(mask_region.convert('RGB'))

mask_hsv = rgb2hsv(mask_region_np)
mask_sat = mask_hsv[:, :, 1]
sat_threshold = np.float64(0.04)
pixel_tissue = mask_sat > sat_threshold
pixel_tissue = binary_closing(pixel_tissue, disk(5))
pixel_tissue = binary_fill_holes(pixel_tissue)
pixel_tissue = binary_opening(pixel_tissue, disk(3))

# Resize to exact canvas size
tissue_mask_full = np.array(
    Image.fromarray(pixel_tissue.astype(np.uint8) * 255).resize(
        (out_w, out_h), Image.NEAREST
    )
) > 127

print(f"\nOtsu saturation threshold: {sat_threshold:.3f}")
print(f"Pixel-level mask: {tissue_mask_full.shape} == canvas ({out_h}, {out_w})")
print(f"Tissue pixel ratio: {tissue_mask_full.mean()*100:.1f}%")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(mask_region_np)
axes[0].set_title('WSI Region for Mask', fontsize=13, fontweight='bold')
axes[0].axis('off')
axes[1].imshow(mask_sat, cmap='hot')
axes[1].set_title(f'Saturation (Otsu={sat_threshold:.3f})', fontsize=13, fontweight='bold')
axes[1].axis('off')
mask_display = Image.fromarray(tissue_mask_full.astype(np.uint8) * 255).resize(
    (400, int(400 * out_h / out_w)), Image.NEAREST)
axes[2].imshow(np.array(mask_display), cmap='gray')
axes[2].set_title(f'Pixel Mask ({out_w}x{out_h})', fontsize=13, fontweight='bold')
axes[2].axis('off')
plt.tight_layout()
plt.show()

## 4. Patch-wise Virtual Staining with Overlap Blending (IHC → HE)

Overlap-tile 방식으로 패치를 겹쳐서 추출하고, 겹치는 영역은 linear blending으로 부드럽게 합성하여 격자 아티팩트를 제거합니다.

In [ ]:
transform = transforms.Compose([
    transforms.Resize(patch_size),
    transforms.ToTensor()
])

# Create 2D raised-cosine blending weight
def make_blend_weight(size, overlap):
    w = np.ones(size, dtype=np.float64)
    if overlap > 0:
        ramp = np.linspace(0, np.pi / 2, overlap)
        fade = np.sin(ramp) ** 2
        w[:overlap] *= fade
        w[-overlap:] *= fade[::-1]
    return np.outer(w, w)

blend_weight = make_blend_weight(patch_size, overlap)

# Float canvases for weighted blending
output_acc = np.zeros((out_h, out_w, 3), dtype=np.float64)
input_acc = np.zeros((out_h, out_w, 3), dtype=np.float64)
weight_acc = np.zeros((out_h, out_w), dtype=np.float64)

tissue_count = 0
total_patches = n_patches_x * n_patches_y

def is_tissue_at_pixel(px, py, ps, threshold=0.1):
    """pixel-level mask로 tissue 여부 판단 (canvas 좌표 기준)"""
    block = tissue_mask_full[py:py+ps, px:px+ps]
    if block.size == 0:
        return False
    return block.mean() > threshold

with torch.no_grad():
    for yi in tqdm(range(n_patches_y), desc="Processing rows"):
        for xi in range(n_patches_x):
            # Canvas 좌표: 인덱스 기반 (정수 연산, 오차 없음)
            px = xi * stride
            py = yi * stride
            # Level-0 좌표: slide에서 읽기용
            x0 = positions_x[xi]
            y0 = positions_y[yi]

            region = slide.read_region((x0, y0), best_level, (level_read_size, level_read_size))
            region = region.convert('RGB')
            region = region.resize((patch_size, patch_size), Image.LANCZOS)
            region_np = np.array(region, dtype=np.float64)

            input_acc[py:py+patch_size, px:px+patch_size] += region_np * blend_weight[:, :, None]

            # tissue_mask_full(pixel-level) 기준으로 skip 판단
            if not is_tissue_at_pixel(px, py, patch_size):
                output_acc[py:py+patch_size, px:px+patch_size] += region_np * blend_weight[:, :, None]
                weight_acc[py:py+patch_size, px:px+patch_size] += blend_weight
                continue

            tissue_count += 1

            img_tensor = transform(region) * 2.0 - 1.0
            img_tensor = img_tensor.unsqueeze(0).to(device)
            fake_he = F(img_tensor)

            fake_he = (fake_he[0].cpu() * 0.5 + 0.5).clamp(0, 1)
            fake_he_np = (fake_he.permute(1, 2, 0).numpy() * 255).astype(np.float64)

            output_acc[py:py+patch_size, px:px+patch_size] += fake_he_np * blend_weight[:, :, None]
            weight_acc[py:py+patch_size, px:px+patch_size] += blend_weight

# Normalize
uncovered = weight_acc < 0.01
weight_acc = np.maximum(weight_acc, 1e-10)
output_canvas = (output_acc / weight_acc[:, :, None]).clip(0, 255).astype(np.uint8)
input_canvas = (input_acc / weight_acc[:, :, None]).clip(0, 255).astype(np.uint8)

# Uncovered edge pixels → white
output_canvas[uncovered] = 255
input_canvas[uncovered] = 255

# Background pixels → original input color
output_canvas[~tissue_mask_full] = input_canvas[~tissue_mask_full]

print(f"\nProcessed {tissue_count}/{total_patches} tissue patches ({tissue_count/total_patches*100:.1f}%)")
print(f"Canvas: {output_canvas.shape}, Mask: {tissue_mask_full.shape}")

## 5. Thumbnail Visualization

Display input IHC and generated HE as thumbnails side by side.

In [ ]:
# Create thumbnail-sized images
thumb_width = 1500
aspect_ratio = input_canvas.shape[0] / input_canvas.shape[1]
thumb_height = int(thumb_width * aspect_ratio)

input_thumb = Image.fromarray(input_canvas).resize((thumb_width, thumb_height), Image.LANCZOS)
output_thumb = Image.fromarray(output_canvas).resize((thumb_width, thumb_height), Image.LANCZOS)

# Real HnE thumbnail (from paired WSI)
hne_thumb = slide_hne.get_thumbnail((thumb_width, thumb_height))

slide_name = os.path.basename(wsi_path).replace(f'.{params["img_form"]}', '')

# Plot: 3 images - Input IHC, Generated HE, Real HE
fig, axes = plt.subplots(3, 1, figsize=(int(20 * aspect_ratio), 24))
fig.suptitle(f'WSI Virtual Staining: IHC → HE\n{slide_name}', 
             fontsize=18, fontweight='bold', y=1.01)

axes[0].imshow(input_thumb)
axes[0].set_title('Input (IHC)', fontsize=16, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(output_thumb)
axes[1].set_title('Generated (HE) - CycleGAN', fontsize=16, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(hne_thumb)
axes[2].set_title('Real HE (Consecutive Section)', fontsize=16, fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
save_path = f'{result_dir}/wsis/{slide_name}_virtual_staining.png'
create_dir(os.path.dirname(f'{result_dir}/wsis/'))
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved to: {save_path}")

## 6. Zoomed-in Patch Comparison

In [ ]:
# Show random tissue patch comparisons
num_examples = 6

# Find tissue patch locations from pixel mask
tissue_patches = []
for yi in range(n_patches_y):
    for xi in range(n_patches_x):
        px, py = xi * stride, yi * stride
        if is_tissue_at_pixel(px, py, patch_size):
            tissue_patches.append((yi, xi, py, px))

# Random sample
np.random.seed(42)
sample_indices = np.random.choice(len(tissue_patches), min(num_examples, len(tissue_patches)), replace=False)

fig, axes = plt.subplots(2, num_examples, figsize=(5 * num_examples, 10))
fig.suptitle('Patch-level Comparison (Full Resolution)', fontsize=16, fontweight='bold')

for i, idx in enumerate(sample_indices):
    _, _, py, px = tissue_patches[idx]
    
    input_patch = input_canvas[py:py+patch_size, px:px+patch_size]
    output_patch = output_canvas[py:py+patch_size, px:px+patch_size]
    
    axes[0, i].imshow(input_patch)
    axes[0, i].set_title(f'IHC', fontsize=11)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(output_patch)
    axes[1, i].set_title(f'Generated HE', fontsize=11)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Input IHC', fontsize=13, fontweight='bold')
axes[1, 0].set_ylabel('Generated HE', fontsize=13, fontweight='bold')

plt.tight_layout()
patch_path = f'{result_dir}/patches/{slide_name}_patch_comparison.png'
create_dir(os.path.dirname(f'{result_dir}/patches/'))
plt.savefig(patch_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved to: {patch_path}")
slide.close()
slide_hne.close()

## 7. Batch Processing: All WSIs

In [ ]:
from skimage.color import rgb2hsv
from skimage.morphology import binary_closing, binary_opening, disk
from scipy.ndimage import binary_fill_holes

transform = transforms.Compose([
    transforms.Resize(params['input_size']),
    transforms.ToTensor()
])

patch_size = params['input_size']
overlap = patch_size // 4
stride = patch_size - overlap
sat_threshold = np.float64(0.04)

def make_blend_weight(size, overlap):
    w = np.ones(size, dtype=np.float64)
    if overlap > 0:
        ramp = np.linspace(0, np.pi / 2, overlap)
        fade = np.sin(ramp) ** 2
        w[:overlap] *= fade
        w[-overlap:] *= fade[::-1]
    return np.outer(w, w)

blend_weight = make_blend_weight(patch_size, overlap)

create_dir(f'{result_dir}/wsis')
create_dir(f'{result_dir}/patches')

# All IHC WSIs
all_ihc = sorted(glob(f'{data_dir_ihc}*.{params["img_form"]}'))
print(f"Total WSIs to process: {len(all_ihc)}\n")

for wsi_idx, wsi_path in enumerate(all_ihc):
    slide_name = os.path.basename(wsi_path).replace(f'.{params["img_form"]}', '')
    sample_id = re.search(r'SS-(\d+)', os.path.basename(wsi_path)).group(1)
    
    # Skip if already processed
    save_path = f'{result_dir}/wsis/{slide_name}_virtual_staining.png'
    if os.path.exists(save_path):
        print(f"[{wsi_idx+1}/{len(all_ihc)}] SKIP (exists): {slide_name}")
        continue
    
    # Check HnE pair
    if sample_id not in hne_dict:
        print(f"[{wsi_idx+1}/{len(all_ihc)}] SKIP (no HnE): {slide_name}")
        continue
    
    hne_path = hne_dict[sample_id]
    print(f"[{wsi_idx+1}/{len(all_ihc)}] Processing: {slide_name}")
    
    try:
        # Open slides
        slide = openslide.OpenSlide(wsi_path)
        slide_hne = openslide.OpenSlide(hne_path)
        
        native_mpp = float(slide.properties.get('openslide.mpp-x', 0.25))
        downsample_factor = params['target_mpp'] / native_mpp
        W, H = slide.dimensions
        
        read_size = int(patch_size * downsample_factor)
        best_level = slide.get_best_level_for_downsample(downsample_factor)
        level_downsample = slide.level_downsamples[best_level]
        level_read_size = int(read_size / level_downsample)
        read_stride = int(stride * downsample_factor)
        
        positions_x = list(range(0, W - read_size + 1, read_stride))
        positions_y = list(range(0, H - read_size + 1, read_stride))
        n_px, n_py = len(positions_x), len(positions_y)
        
        out_w = (n_px - 1) * stride + patch_size
        out_h = (n_py - 1) * stride + patch_size
        canvas_l0_w = positions_x[-1] + read_size
        canvas_l0_h = positions_y[-1] + read_size
        
        # --- Tissue mask ---
        mask_level = slide.get_best_level_for_downsample(canvas_l0_w / (out_w // 4))
        mask_level_ds = slide.level_downsamples[mask_level]
        mask_rw = int(canvas_l0_w / mask_level_ds)
        mask_rh = int(canvas_l0_h / mask_level_ds)
        mask_region = slide.read_region((0, 0), mask_level, (mask_rw, mask_rh))
        mask_np = np.array(mask_region.convert('RGB'))
        
        mask_sat = rgb2hsv(mask_np)[:, :, 1]
        pixel_tissue = mask_sat > sat_threshold
        pixel_tissue = binary_closing(pixel_tissue, disk(5))
        pixel_tissue = binary_fill_holes(pixel_tissue)
        pixel_tissue = binary_opening(pixel_tissue, disk(3))
        
        tissue_mask_full = np.array(
            Image.fromarray(pixel_tissue.astype(np.uint8) * 255).resize(
                (out_w, out_h), Image.NEAREST)
        ) > 127
        
        # --- Inference with overlap blending ---
        output_acc = np.zeros((out_h, out_w, 3), dtype=np.float64)
        input_acc = np.zeros((out_h, out_w, 3), dtype=np.float64)
        weight_acc = np.zeros((out_h, out_w), dtype=np.float64)
        tissue_count = 0
        
        with torch.no_grad():
            for yi in tqdm(range(n_py), desc=f"  [{wsi_idx+1}] Rows", leave=False):
                for xi in range(n_px):
                    px, py = xi * stride, yi * stride
                    x0, y0 = positions_x[xi], positions_y[yi]
                    
                    region = slide.read_region((x0, y0), best_level, (level_read_size, level_read_size))
                    region = region.convert('RGB').resize((patch_size, patch_size), Image.LANCZOS)
                    region_np = np.array(region, dtype=np.float64)
                    
                    input_acc[py:py+patch_size, px:px+patch_size] += region_np * blend_weight[:, :, None]
                    
                    # Tissue check
                    block = tissue_mask_full[py:py+patch_size, px:px+patch_size]
                    if block.size == 0 or block.mean() <= 0.1:
                        output_acc[py:py+patch_size, px:px+patch_size] += region_np * blend_weight[:, :, None]
                        weight_acc[py:py+patch_size, px:px+patch_size] += blend_weight
                        continue
                    
                    tissue_count += 1
                    img_tensor = (transform(region) * 2.0 - 1.0).unsqueeze(0).to(device)
                    fake_he = F(img_tensor)
                    fake_np = ((fake_he[0].cpu() * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0).numpy() * 255).astype(np.float64)
                    
                    output_acc[py:py+patch_size, px:px+patch_size] += fake_np * blend_weight[:, :, None]
                    weight_acc[py:py+patch_size, px:px+patch_size] += blend_weight
        
        # Normalize
        uncovered = weight_acc < 0.01
        weight_acc = np.maximum(weight_acc, 1e-10)
        output_canvas = (output_acc / weight_acc[:, :, None]).clip(0, 255).astype(np.uint8)
        input_canvas = (input_acc / weight_acc[:, :, None]).clip(0, 255).astype(np.uint8)
        output_canvas[uncovered] = 255
        input_canvas[uncovered] = 255
        output_canvas[~tissue_mask_full] = input_canvas[~tissue_mask_full]
        
        # --- Save thumbnail comparison ---
        thumb_width = 1500
        ar = input_canvas.shape[0] / input_canvas.shape[1]
        thumb_height = int(thumb_width * ar)
        
        input_thumb = Image.fromarray(input_canvas).resize((thumb_width, thumb_height), Image.LANCZOS)
        output_thumb = Image.fromarray(output_canvas).resize((thumb_width, thumb_height), Image.LANCZOS)
        hne_thumb = slide_hne.get_thumbnail((thumb_width, thumb_height))
        
        fig, axes = plt.subplots(3, 1, figsize=(int(20 * ar), 24))
        fig.suptitle(f'WSI Virtual Staining: IHC → HE\n{slide_name}',
                     fontsize=18, fontweight='bold', y=1.01)
        axes[0].imshow(input_thumb); axes[0].set_title('Input (IHC)', fontsize=16, fontweight='bold'); axes[0].axis('off')
        axes[1].imshow(output_thumb); axes[1].set_title('Generated (HE) - CycleGAN', fontsize=16, fontweight='bold'); axes[1].axis('off')
        axes[2].imshow(hne_thumb); axes[2].set_title('Real HE (Consecutive Section)', fontsize=16, fontweight='bold'); axes[2].axis('off')
        plt.tight_layout()
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close()
        
        # --- Save patch comparison ---
        tissue_patches = [(yi, xi, yi*stride, xi*stride)
                          for yi in range(n_py) for xi in range(n_px)
                          if tissue_mask_full[yi*stride:yi*stride+patch_size, xi*stride:xi*stride+patch_size].mean() > 0.1]
        
        if len(tissue_patches) >= 6:
            np.random.seed(42)
            sample_idx = np.random.choice(len(tissue_patches), 6, replace=False)
            fig, axes = plt.subplots(2, 6, figsize=(30, 10))
            fig.suptitle(f'Patch Comparison: {slide_name}', fontsize=16, fontweight='bold')
            for i, idx in enumerate(sample_idx):
                _, _, py, px = tissue_patches[idx]
                axes[0, i].imshow(input_canvas[py:py+patch_size, px:px+patch_size]); axes[0, i].set_title('IHC'); axes[0, i].axis('off')
                axes[1, i].imshow(output_canvas[py:py+patch_size, px:px+patch_size]); axes[1, i].set_title('Gen HE'); axes[1, i].axis('off')
            plt.tight_layout()
            plt.savefig(f'{result_dir}/patches/{slide_name}_patch_comparison.png', dpi=150, bbox_inches='tight')
            plt.close()
        
        slide.close()
        slide_hne.close()
        
        total = n_px * n_py
        print(f"  Done: {tissue_count}/{total} tissue patches ({tissue_count/total*100:.0f}%), canvas {out_w}x{out_h}")
        
        # Free memory
        del output_acc, input_acc, weight_acc, output_canvas, input_canvas, tissue_mask_full
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
        
    except Exception as e:
        print(f"  ERROR: {e}")
        try:
            slide.close(); slide_hne.close()
        except:
            pass
        continue

print(f"\nAll done! Results saved to: {result_dir}")